In [ ]:
import tensorflow as tf
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

2026-04-16 14:34:25.152997: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776350065.391156      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776350065.454383      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776350065.999131      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776350065.999177      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776350065.999180      55 computation_placer.cc:177] computation placer alr

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_path = "/kaggle/input/datasets/iarunava/cell-images-for-detecting-malaria/cell_images/cell_images"

# Training — augmentation + rescale
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=180,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)
train_generator = train_datagen.flow_from_directory(
    base_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42
)

# Validation — rescale only, NO augmentation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)
val_generator = val_datagen.flow_from_directory(
    base_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print(train_generator.class_indices)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger

callbacks = [
    ModelCheckpoint("vgg19_malaria.keras", monitor="val_accuracy",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    CSVLogger("vgg19_malaria_log.csv")
]

In [ ]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Reshape, Multiply
from tensorflow.keras.models import Sequential

In [ ]:
def squeeze_excite_block(input_tensor, ratio=8):
    filters = input_tensor.shape[-1]
    se = GlobalAveragePooling2D()(input_tensor)
    se = Dense(filters // ratio, activation='relu',
               kernel_initializer='he_normal', use_bias=False)(se)
    se = Dense(filters, activation='sigmoid',
               kernel_initializer='he_normal', use_bias=False)(se)
    se = Reshape((1, 1, filters))(se)
    return Multiply()([input_tensor, se])

In [ ]:
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-8:]:
    layer.trainable = True

# Insert SE after last conv of each block
SE_TARGETS = {'block4_conv4', 'block5_conv4'}

x = base_model.input
for layer in base_model.layers[1:]:
    x = layer(x)
    if layer.name in SE_TARGETS:
        x = squeeze_excite_block(x, ratio=16)

x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(train_generator.num_classes, activation='softmax')(x)

se_vgg19 = Model(inputs=base_model.input, outputs=outputs, name="SE_VGG19")

In [ ]:
se_vgg19.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam

se_vgg19.compile(optimizer=Adam(1e-4),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

history = se_vgg19.fit(train_generator, validation_data=val_generator,
                       epochs=15, callbacks=callbacks)


In [ ]:
for layer in se_vgg19.layers:
    layer.trainable = True

# Keep early blocks frozen
for layer in base_model.layers[:-5]:
    layer.trainable = False

In [11]:
se_vgg19.compile(optimizer=Adam(1e-5),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

callbacks_ft = callbacks[:-1] + [CSVLogger("log.csv", append=True)]
history_ft = se_vgg19.fit(train_generator, validation_data=val_generator,
                          epochs=10, callbacks=callbacks_ft)

Epoch 1/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step - accuracy: 0.9394 - loss: 0.1830
Epoch 1: val_accuracy improved from 0.93013 to 0.93140, saving model to vgg19_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 326s 463ms/step - accuracy: 0.9394 - loss: 0.1830 - val_accuracy: 0.9314 - val_loss: 0.2039 - learning_rate: 1.0000e-05
Epoch 2/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - accuracy: 0.9447 - loss: 0.1643
Epoch 2: val_accuracy improved from 0.93140 to 0.93793, saving model to vgg19_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 303s 440ms/step - accuracy: 0.9447 - loss: 0.1643 - val_accuracy: 0.9379 - val_loss: 0.1772 - learning_rate: 1.0000e-05
Epoch 3/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 418ms/step - accuracy: 0.9493 - loss: 0.1468
Epoch 3: val_accuracy improved from 0.93793 to 0.94011, saving model to vgg19_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 304s 441ms/step - accuracy: 0.9493 - loss: 0.1468 - val_accuracy: 0.9401 - val_loss: 0.1779 - learning_rate: 1.0000e-05
Epoch 4/10
68

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

val_generator.reset()
pred_probs = se_vgg19.predict(val_generator, verbose=1)

# ✅ Correct for categorical/softmax output
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = val_generator.classes

In [ ]:
# Combined training curve
plt.figure(figsize=(10, 4))

all_acc = history.history['accuracy'] + history_ft.history['accuracy']
all_val = history.history['val_accuracy'] + history_ft.history['val_accuracy']

plt.plot(all_acc, label='Train Accuracy')
plt.plot(all_val, label='Val Accuracy')
plt.axvline(x=len(history.history['accuracy'])-1,
            color='gray', linestyle='--', label='Fine-tune start')
plt.legend()
plt.title('SE-VGG19 Training History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.show()